In [1]:
import pandas as pd
import threading
import time
import os

# Simulação de Streaming dos novos dados

# Processamento das Vistas de Tempo Real

# Reconstrução das vistas

In [ ]:
import pandas as pd
import os
import threading
import time

finaliza_laco = False
estado_stream = {}

lock_arquivo = threading.Lock()



metricas = ["media","contagem","maximo","minimo","mediana","desvio_padrao","variancia","moda"]

# ================================
# TEMPO REAL (INCREMENTAL)
# ================================
def atualizar_vista_rapida(linha, header, var, fazer):
    global estado_stream

    caminho = "vistas_tempo_real/contagem_incremental.csv"

    valores = linha.split(",")

    try:
        idx = header.index(var)
        valor = float(valores[idx])
    except:
        return

    data = valores[0]
    chave = (data, var)

    if chave not in estado_stream:
        estado_stream[chave] = {
            "count": 0,
            "soma": 0,
            "max": valor,
            "min": valor,
            "valores": []
        }

    estado = estado_stream[chave]

    # Atualizando os valores
    estado["count"] += 1
    estado["soma"] += valor
    estado["max"] = max(estado["max"], valor)
    estado["min"] = min(estado["min"], valor)
    estado["valores"].append(valor)

    # Calculando o resultado
    if fazer == "media":
        resultado = estado["soma"] / estado["count"]
    elif fazer == "contagem":
        resultado = estado["count"]
    elif fazer == "maximo":
        resultado = estado["max"]
    elif fazer == "minimo":
        resultado = estado["min"]
    elif fazer == "mediana":
        resultado = pd.Series(estado["valores"]).median()
    elif fazer == "desvio_padrao":
        resultado = pd.Series(estado["valores"]).std()
    elif fazer == "variancia":
        resultado = pd.Series(estado["valores"]).var()
    elif fazer == "moda":
        resultado = pd.Series(estado["valores"]).mode()[0]
    else:
        return

    os.makedirs("vistas_tempo_real", exist_ok=True)

    with lock_arquivo:

        df = pd.read_csv(caminho)

        df["data"] = df["data"].astype(str)
        data = str(data)

        mascara = (df["data"] == data) & (df["var"] == var)

        # garante colunas
        for m in metricas:
            if m not in df.columns:
                df[m] = None

        if mascara.any():
            df.loc[mascara, fazer] = resultado
        else:
            nova_linha = {
                "data": data,
                "var": var,
                **{m: None for m in metricas}
            }
            nova_linha[fazer] = resultado

            df = pd.concat([df, pd.DataFrame([nova_linha])], ignore_index=True)

        df.to_csv(caminho, index=False)


# ================================
# STREAM
# ================================
def stream_dados(arq):
    global finaliza_laco

    while not finaliza_laco:
        linha = arq.readline().strip()

        if not linha:
            time.sleep(0.2)
            continue

        yield linha


def monitora_linhas(arquivo):
    print("Monitorando...")

    with open(arquivo, "r") as arq:
        header = arq.readline().strip().split(",")

        for linha in stream_dados(arq):
            atualizar_vista_rapida(linha, header, "duration_(secs)", "media")


def simular_stream_csv(entrada, saida, delay=0.05):
    global finaliza_laco

    os.makedirs(os.path.dirname(saida), exist_ok=True)

    with open(entrada, "r") as arq_in:
        with open(saida, "w") as arq_out:

            header = arq_in.readline()
            arq_out.write(header)
            arq_out.flush()

            for linha in arq_in:
                if finaliza_laco:
                    break

                arq_out.write(linha)
                arq_out.flush()
                time.sleep(delay)


# ================================
# LOTE (RECONSTRUÇÃO)
# ================================
def reconstruir_vistas_lote():
    print("Reconstruindo vistas de lote...")

    entrada = "vistas_tempo_real/contagem_incremental.csv"
    saida = "vistas_lote/vistas_de_lote.csv"

    os.makedirs("vistas_lote", exist_ok=True)

    if not os.path.exists(entrada):
        print("Sem dados de tempo real.")
        return

    df = pd.read_csv(entrada)

    if df.empty:
        print("CSV de tempo real vazio.")
        return

    # pega última linha (estado final)
    ultima = df.tail(1).copy()

    ultima['data'] = pd.to_datetime(ultima['data']).dt.date

    if not os.path.exists(saida):
        ultima.to_csv(saida, index=False)
    else:
        df_lote = pd.read_csv(saida)
        df_final = pd.concat([df_lote, ultima], ignore_index=True)
        df_final.to_csv(saida, index=False)

    print("Lote atualizado com sucesso!")


# ================================
# MAIN
# ================================
if __name__ == "__main__":

    arquivo_entrada = "dados_brutos/2017-03-21.csv"
    arquivo_stream = "dados_novos/fluxo.log"
    csv_tempo_real = "vistas_tempo_real/contagem_incremental.csv"

    # cria pastas
    os.makedirs("dados_novos", exist_ok=True)
    os.makedirs("vistas_tempo_real", exist_ok=True)

    # 🔥 cria CSV com schema correto
    df_init = pd.DataFrame(columns=["data", "var"] + metricas)
    df_init.to_csv(csv_tempo_real, index=False)

    # garante arquivo stream
    open(arquivo_stream, "w").close()

    t1 = threading.Thread(target=simular_stream_csv, args=(arquivo_entrada, arquivo_stream), daemon=True)
    t2 = threading.Thread(target=monitora_linhas, args=(arquivo_stream,), daemon=True)

    t1.start()
    t2.start()

    try:
        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        finaliza_laco = True

        t1.join()
        t2.join()

        print("Encerrado.")

        # atualiza lote ao final
        reconstruir_vistas_lote()

Monitorando...


C:\Users\edson\AppData\Local\Temp\ipykernel_8464\2517174840.py:96: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([nova_linha])], ignore_index=True)


Encerrado.
Reconstruindo vistas de lote...
Lote atualizado com sucesso!
